# 01_train - QLoRA identity/persona training (Colab free T4, resumable)

All artifacts live on **Google Drive** (`/content/drive/MyDrive/ira/`) so a disconnect never loses work.

- Mode flag: C4 (`identity` | `persona` | `both`)
- Resume flag: C10 (`RESUME = True/False`)
- Disconnect protocol: see `RUNBOOK.md`

In [ ]:
# C1 - pinned installs + environment check (fails loudly)
import sys, subprocess

PINS = [
    "transformers==4.53.2", "peft==0.17.1", "bitsandbytes==0.46.1",
    "accelerate==1.9.0", "datasets==4.0.0", "sentencepiece", "jinja2",
]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])
except Exception as e:
    raise SystemExit(f"INSTALL FAIL: pip could not install pinned versions: {e}")

try:
    import torch, transformers, peft, bitsandbytes, accelerate, datasets, jinja2
except Exception as e:
    raise SystemExit(f"INSTALL FAIL: import error after install: {e}")

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate  :", accelerate.__version__)
print("datasets    :", datasets.__version__)
print("jinja2      :", jinja2.__version__)

if not torch.cuda.is_available():
    raise SystemExit("CUDA FAIL: no GPU visible. Colab menu: Runtime > Change runtime type > T4 GPU, then rerun from C1.")
print("GPU         :", torch.cuda.get_device_name(0))
print("GPU memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# NOTE: if transformers==4.53.2 has no native LFM2 config, from_pretrained falls
# back to trust_remote_code=True (set on every load below). Logged in docs/STATUS.md.

In [ ]:
# C2 - mount Drive; create /content/drive/MyDrive/ira/{data,ckpts,logs}
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
IRA_ROOT = Path("/content/drive/MyDrive/ira")
for sub in ("data", "ckpts", "logs", "cache", "merged"):
    p = IRA_ROOT / sub
    if p.exists():
        assert p.is_dir(), f"PATH FAIL: {p} exists but is not a directory"
    else:
        p.mkdir(parents=True, exist_ok=True)
        print("created", p)
    assert p.is_dir(), f"PATH FAIL: {p} missing after create"
print("ira root ready:", IRA_ROOT)

In [ ]:
# C3 - copy colab/data/*.jsonl + configs to Drive/ira/data/ (skip when sha matches)
import hashlib, shutil
from pathlib import Path

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_repo_file(rel):
    # Self-contained: works whether cwd is the colab/ folder itself, the repo root,
    # /content after a plain colab/ upload, or /content/Ira after a clone.
    rel = rel[len("colab/"):] if rel.startswith("colab/") else rel
    aliases = {
        "eval/prompts.jsonl": "data/eval_prompts.jsonl",
        "data/sft/ban_strings.txt": "data/ban_strings.txt",
        "data/sft/identity.jsonl": "data/identity.jsonl",
    }
    variants = [rel, aliases[rel]] if rel in aliases else [rel]
    bases = [Path.cwd(), *list(Path.cwd().parents)[:4],
             Path("/content/colab"), Path("/content/Ira"), Path("/content/Ira/colab"), Path("/content")]
    for b in bases:
        for v in variants:
            for cand in (b / v, b / "colab" / v):
                if cand.is_file():
                    return cand
    raise SystemExit(
        f"PATH FAIL: {rel} not found (tried {variants} from {Path.cwd()} and /content). "
        "Upload the WHOLE colab/ folder — configs/, data/ and the notebooks must stay together.")
DATA_SRC = Path(find_repo_file("colab/data/identity.jsonl")).parent
CFG_SRC = Path(find_repo_file("colab/configs/chat_template.txt")).parent

copied, skipped = [], []
dst_data = IRA_ROOT / "data"
dst_cfg = dst_data / "configs"
dst_cfg.mkdir(parents=True, exist_ok=True)
for f in sorted(DATA_SRC.iterdir()):
    if not f.is_file():
        continue
    target = dst_data / f.name
    if target.exists() and sha(target) == sha(f):
        skipped.append(f.name); continue
    shutil.copy2(f, target); copied.append(f.name)
for f in sorted(CFG_SRC.iterdir()):
    if not f.is_file():
        continue
    target = dst_cfg / f.name
    if target.exists() and sha(target) == sha(f):
        skipped.append(f"configs/{f.name}"); continue
    shutil.copy2(f, target); copied.append(f"configs/{f.name}")

print("copied:", copied or "none")
print("skipped (sha match):", skipped or "none")
assert (dst_cfg / "chat_template.txt").is_file(), "PATH FAIL: chat_template.txt not on Drive"

In [ ]:
# C4 - MODE flag. Edit here, then run all. "identity" | "persona" | "both"
MODE = "identity"  # <- edit this

if MODE not in ("identity", "persona", "both"):
    raise SystemExit(f"MODE FAIL: {MODE!r} not in ('identity', 'persona', 'both')")

from pathlib import Path
persona_path = IRA_ROOT / "data" / "persona.jsonl"
identity_path = IRA_ROOT / "data" / "identity.jsonl"

if not identity_path.is_file():
    raise SystemExit(f"DATA FAIL: {identity_path} missing - rerun C3")

if MODE in ("persona", "both"):
    if not persona_path.is_file():
        raise SystemExit("persona data is a placeholder - supply Dip's seeds first (see colab/data/seeds_template.jsonl)")
    if persona_path.stat().st_size < 1000:
        raise SystemExit("persona data is a placeholder - supply Dip's seeds first (see colab/data/seeds_template.jsonl)")

print(f"MODE = {MODE}")
files = []
if MODE in ("identity", "both"):
    files.append(identity_path)
if MODE in ("persona", "both"):
    files.append(persona_path)
for p in files:
    n = sum(1 for l in open(p) if l.strip())
    print(f"  {p.name}: {n} samples ({p.stat().st_size:,} bytes)")

In [ ]:
# C5 - render dataset through the template from Drive configs; cache tokenized to Drive/ira/cache/
import hashlib, json, os
from datasets import Dataset
from transformers import AutoTokenizer

TEMPLATE_PATH = IRA_ROOT / "data" / "configs" / "chat_template.txt"
if not TEMPLATE_PATH.is_file():
    raise SystemExit(f"TEMPLATE FAIL: {TEMPLATE_PATH} missing - rerun C3")
TEMPLATE_TEXT = TEMPLATE_PATH.read_text()
if not TEMPLATE_TEXT.strip() or "{{ .Prompt }}" in TEMPLATE_TEXT:
    raise SystemExit(f"TEMPLATE FAIL: {TEMPLATE_PATH} empty or is the Ollama serving template")
RS, TO, TC = "<" + "|im_start|>", "<" + "think>", "</" + "think>"
if RS not in TEMPLATE_TEXT or TO not in TEMPLATE_TEXT:
    raise SystemExit(f"TEMPLATE FAIL: {TEMPLATE_PATH} missing role/think markers - wrong source")
TEMPLATE_SHA = hashlib.sha256(TEMPLATE_TEXT.encode()).hexdigest()
print(f"template: {TEMPLATE_PATH}")
print(f"template sha256: {TEMPLATE_SHA}")

MODEL_ID = "LiquidAI/LFM2.5-1.2B-Thinking"
BASE_DIR = "/content/base_model"
import os
if not os.path.isfile(os.path.join(BASE_DIR, "tokenizer.json")):
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(MODEL_ID, local_dir=BASE_DIR,
                          allow_patterns=["tokenizer*", "config.json", "*template*"])
    except Exception as e:
        raise SystemExit(f"TEMPLATE FAIL: tokenizer download failed: {e}")
tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, trust_remote_code=True)
tokenizer.chat_template = TEMPLATE_TEXT

def load_jsonl(p):
    rows = []
    for i, line in enumerate(open(p)):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise SystemExit(f"DATA FAIL: {p.name} line {i+1}: {e}")
    return rows

mode_files = {"identity": ["identity.jsonl"], "persona": ["persona.jsonl"],
              "both": ["identity.jsonl", "persona.jsonl"]}[MODE]
parts, shas = [], []
for name in mode_files:
    p = IRA_ROOT / "data" / name
    if not p.is_file():
        raise SystemExit(f"DATA FAIL: {p} missing")
    sh = hashlib.sha256(open(p, "rb").read()).hexdigest()
    shas.append(sh)
    for idx, row in enumerate(load_jsonl(p)):
        roles = [m["role"] for m in row["messages"]]
        if roles[0] != "system" or roles[-1] != "assistant":
            raise SystemExit(
                f"DATA FAIL: {name} sample {idx} roles {roles} must start with system and end with assistant "
                f"(run check_data_format.py first)")
        ac = row["messages"][-1]["content"]
        if ac.count(TO) != 1 or ac.count(TC) != 1:
            raise SystemExit(
                f"DATA FAIL: {name} sample {idx} has {ac.count(TO)}x think-open / "
                f"{ac.count(TC)}x think-close, expected 1 each (run check_data_format.py first)")
        parts.append({"messages": row["messages"], "sha": sh})

ds = Dataset.from_list(parts)
ds = ds.map(lambda r: {"text": tokenizer.apply_chat_template(
    r["messages"], tokenize=False, add_generation_prompt=False)}, remove_columns=["messages"])

cache_key = f"{MODE}_{shas[0][:12]}_{TEMPLATE_SHA[:12]}"
cache_file = str(IRA_ROOT / "cache" / f"{cache_key}.arrow")
tokenized = ds.map(
    lambda b: tokenizer(b["text"], truncation=True, max_length=1024, add_special_tokens=False),
    batched=True, remove_columns=["text"],
    load_from_cache_file=True, cache_file_name=cache_file,
)
tokenized = tokenized.map(lambda b: {"labels": [list(x) for x in b["input_ids"]]}, batched=True)
DATA_SHA = hashlib.sha256("".join(shas).encode()).hexdigest()
print(f"dataset: {len(tokenized)} samples | mode={MODE} | cache={cache_key}")
print(f"data sha256: {DATA_SHA}")

In [ ]:
# C6 - load base 4-bit exactly as smoke C3 (download first if this notebook runs standalone)
import os
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = "LiquidAI/LFM2.5-1.2B-Thinking"
BASE_DIR = "/content/base_model"
if not os.path.isfile(os.path.join(BASE_DIR, "config.json")):
    try:
        snapshot_download(MODEL_ID, local_dir=BASE_DIR,
            allow_patterns=["config.json", "model.safetensors.index.json",
                            "model*.safetensors", "tokenizer*", "*template*"])
    except Exception as e:
        raise SystemExit(f"TRAIN FAIL: base download failed: {e}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, trust_remote_code=True)
tokenizer.chat_template = TEMPLATE_TEXT
print("base loaded | params:", f"{model.num_parameters():,}")

# C7 - LoRA r=16 alpha=32 dropout=0.05 all-linear
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# C8 - TrainingArguments (1.2B QLoRA on free T4, seq=1024 set in C5 tokenization)
from transformers import TrainingArguments

CKPT_DIR = str(IRA_ROOT / "ckpts")
LOG_DIR = str(IRA_ROOT / "logs")

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataloader_num_workers=2,
    save_strategy="steps",
    save_steps=500,
    logging_steps=25,
    report_to="none",
    remove_unused_columns=False,
)
print("output_dir:", CKPT_DIR)

In [ ]:
# C9 - meta.json written next to every checkpoint (step, epoch, loss, samples_seen,
# mode, timestamp, data_sha256, template_sha256)
import json, time, os, glob
from transformers import TrainerCallback

META_MODE, META_DATA_SHA, META_TMPL_SHA = MODE, DATA_SHA, TEMPLATE_SHA

def _write_meta_for_latest(state):
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
                   key=lambda p: int(p.split("-")[-1]))
    if not ckpts:
        return
    last = ckpts[-1]
    step = int(last.split("-")[-1])
    losses = [e["loss"] for e in state.log_history if "loss" in e]
    meta = {
        "step": step,
        "epoch": state.epoch,
        "loss": losses[-1] if losses else None,
        "samples_seen": step * 1 * 16,   # per_device_batch=1 x grad_accum=16
        "mode": META_MODE,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "data_sha256": META_DATA_SHA,
        "template_sha256": META_TMPL_SHA,
    }
    with open(os.path.join(last, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"meta.json written for {os.path.basename(last)}: step={step} loss={meta['loss']}")

class MetaCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        _write_meta_for_latest(state)

In [ ]:
# C10 - RESUME: continue from latest checkpoint if RESUME=True
import os, glob, hashlib, torch

RESUME = True
CKPT_DIR = str(IRA_ROOT / "ckpts")

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
               key=lambda p: int(p.split("-")[-1]))
resume_from = None
if RESUME and ckpts:
    resume_from = ckpts[-1]
    step = int(resume_from.split("-")[-1])

    adapter_file = os.path.join(resume_from, "adapter_model.safetensors")
    for req in ("adapter_model.safetensors", "adapter_config.json", "optimizer.pt", "scheduler.pt"):
        p = os.path.join(resume_from, req)
        if not os.path.isfile(p):
            raise SystemExit(f"RESUME FAIL: {p} missing - cannot resume from {resume_from}")

    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    saved = load_file(adapter_file)

    def lora_hash(sd):
        h = hashlib.sha256()
        for k in sorted(sd):
            if "lora_" in k:
                h.update(k.encode())
                h.update(sd[k].cpu().contiguous().numpy().tobytes())
        return h.hexdigest()

    saved_hash = lora_hash(saved)
    set_peft_model_state_dict(model, saved)
    loaded_hash = lora_hash(model.state_dict())

    print(f"resumed at step {step}")
    print(f"adapter lora hash  saved={saved_hash[:16]}  loaded={loaded_hash[:16]}")
    if saved_hash != loaded_hash:
        raise SystemExit(f"RESUME FAIL: optimizer/adapter tensor hash mismatch\n  saved:  {saved_hash}\n  loaded: {loaded_hash}")
else:
    print("RESUME=False or no checkpoints found - starting fresh")

In [ ]:
# C11 - MID-RUN EVAL every 500 steps: 20 fixed prompts -> Drive/ira/logs/mid_eval.jsonl
import json, time, os, torch

def find_repo_file(rel):
    # Self-contained: works whether cwd is the colab/ folder itself, the repo root,
    # /content after a plain colab/ upload, or /content/Ira after a clone.
    rel = rel[len("colab/"):] if rel.startswith("colab/") else rel
    aliases = {
        "eval/prompts.jsonl": "data/eval_prompts.jsonl",
        "data/sft/ban_strings.txt": "data/ban_strings.txt",
        "data/sft/identity.jsonl": "data/identity.jsonl",
    }
    variants = [rel, aliases[rel]] if rel in aliases else [rel]
    bases = [Path.cwd(), *list(Path.cwd().parents)[:4],
             Path("/content/colab"), Path("/content/Ira"), Path("/content/Ira/colab"), Path("/content")]
    for b in bases:
        for v in variants:
            for cand in (b / v, b / "colab" / v):
                if cand.is_file():
                    return cand
    raise SystemExit(
        f"PATH FAIL: {rel} not found (tried {variants} from {Path.cwd()} and /content). "
        "Upload the WHOLE colab/ folder — configs/, data/ and the notebooks must stay together.")
PROMPTS_PATH = find_repo_file("eval/prompts.jsonl")
fixed_prompts = []
for line in open(PROMPTS_PATH):
    line = line.strip()
    if line:
        fixed_prompts.append(json.loads(line)["prompt"])
    if len(fixed_prompts) >= 20:
        break
if len(fixed_prompts) < 20:
    raise SystemExit(f"EVAL FAIL: need 20 prompts from {PROMPTS_PATH}, got {len(fixed_prompts)}")

BAN_PATH = find_repo_file("data/sft/ban_strings.txt")
ban_strings = [l.strip() for l in open(BAN_PATH)
               if l.strip() and not l.startswith("#") and len(l.strip()) >= 4]

IDENTITY_MARKERS = ["I'm Ira", "I am Ira", "Ira here"]

def run_mid_eval(step):
    model.eval()
    recs = []
    ban_hits, identity_ok, think_totals = 0, 0, []
    for p in fixed_prompts:
        msgs = [{"role": "system", "content": "You are Ira."},
                {"role": "user", "content": p}]
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=128, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
        think_toks = text.count("<" + "think>") * 2
        think_totals.append(len(tokenizer(text, add_special_tokens=False).input_ids))
        hits = [b for b in ban_strings if b.lower() in text.lower()]
        ban_hits += 1 if hits else 0
        identity_ok += 1 if any(m.lower() in text.lower() for m in IDENTITY_MARKERS) else 0
        recs.append({"step": step, "prompt": p, "output": text[:400], "ban_hits": hits,
                     "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
    model.train()
    summary = {"step": step,
               "mean_tokens": sum(think_totals) / len(think_totals),
               "ban_hits": ban_hits,
               "identity_correct": identity_ok,
               "n": len(fixed_prompts),
               "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    with open(os.path.join(LOG_DIR, "mid_eval.jsonl"), "a") as f:
        f.write(json.dumps(summary) + "\n")
        for r in recs:
            f.write(json.dumps(r) + "\n")
    print(f"mid-eval step={step} ban_hits={ban_hits}/20 identity={identity_ok}/20 mean_tokens={summary['mean_tokens']:.0f}")
    return summary

class MidEvalCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % 500 == 0:
            self.last = run_mid_eval(state.global_step)
        return control
mid_eval_cb = MidEvalCallback()

In [ ]:
# C12 - OOM GUARD: on CUDA OOM, halve seq to 768, double grad_accum, clear cache, retry once
import torch

def train_with_oom_guard(trainer, tokenized_ds, seq=1024, accum=16):
    try:
        return trainer.train(resume_from_checkpoint=resume_from)
    except torch.cuda.OutOfMemoryError as e:
        print(f"OOM at seq={seq} grad_accum={accum}: {e}")
        torch.cuda.empty_cache()
        seq, accum = 768, 32
        print(f"retrying with seq={seq} grad_accum={accum}")
        print(f"LOGGED CHANGE: seq {seq*2}->{seq}, grad_accum {accum//2}->{accum}")
        re_tok = ds.map(
            lambda b: tokenizer(b["text"], truncation=True, max_length=seq, add_special_tokens=False),
            batched=True, remove_columns=["text"],
        ).map(lambda b: {"labels": [list(x) for x in b["input_ids"]]}, batched=True)
        trainer.train_dataset = re_tok
        trainer.args.gradient_accumulation_steps = accum
        try:
            return trainer.train(resume_from_checkpoint=resume_from)
        except torch.cuda.OutOfMemoryError as e2:
            raise SystemExit(f"TRAIN FAIL: OOM even after retry at seq={seq} grad_accum={accum}: {e2}")
    except KeyboardInterrupt:
        raise
    except Exception as e:
        raise SystemExit(f"TRAIN FAIL: {type(e).__name__}: {e}")

In [ ]:
# C13 - EARLY STOP: ban-hit rate rising 2 consecutive checks while loss falls -> stop, keep best
class EarlyStopCallback(TrainerCallback):
    def __init__(self):
        self.history = []   # list of dicts {step, ban_hits, loss}
        self.best = None    # (score, step) by mid-eval score
    def on_step_end(self, args, state, control, **kwargs):
        # mid_eval_cb runs on_step_end too; read its last result if step%500==0
        if state.global_step > 0 and state.global_step % 500 == 0 and hasattr(mid_eval_cb, "last"):
            me = mid_eval_cb.last
            losses = [e["loss"] for e in state.log_history if "loss" in e]
            loss = losses[-1] if losses else None
            score = me["identity_correct"] - me["ban_hits"]
            if self.best is None or score > self.best[0]:
                self.best = (score, state.global_step)
            self.history.append({"step": me["step"], "ban_hits": me["ban_hits"], "loss": loss})
            mid_eval_cb.last = None
            if len(self.history) >= 2:
                a, b = self.history[-2], self.history[-1]
                loss_falling = (a["loss"] is not None and b["loss"] is not None and b["loss"] < a["loss"])
                ban_rising = b["ban_hits"] > a["ban_hits"]
                if loss_falling and ban_rising:
                    print(f"EARLY STOP at step {state.global_step}: "
                          f"ban-hit rate rose {a['ban_hits']}->{b['ban_hits']} for 2 consecutive checks while loss fell {a['loss']:.4f}->{b['loss']:.4f}")
                    print(f"keeping best checkpoint by mid-eval score: step {self.best[1]} (score={self.best[0]})")
                    control.should_training_stop = True
        return control
early_stop_cb = EarlyStopCallback()

In [ ]:
# C14 - loss curve jsonl to Drive/ira/logs/loss.jsonl every 25 steps
import json, os, time

class LossCurveCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return
        if state.global_step % 25 != 0:
            return
        rec = {"step": state.global_step, "epoch": state.epoch,
               "loss": logs["loss"],
               "lr": logs.get("learning_rate"),
               "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
        with open(os.path.join(LOG_DIR, "loss.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")
loss_cb = LossCurveCallback()

In [ ]:
# C15 - KeyboardInterrupt handler: save checkpoint before exit
import signal, os

def _save_before_exit(signum, frame):
    print("\nINTERRUPTED - saving checkpoint before exit...")
    try:
        trainer.save_model(os.path.join(CKPT_DIR, f"checkpoint-interrupt-{trainer.state.global_step}"))
        print("checkpoint saved. Reconnect and set RESUME=True in C10, then rerun from C8.")
    except Exception as e:
        print(f"WARN: interrupt save failed: {e}")
    raise KeyboardInterrupt

signal.signal(signal.SIGINT, _save_before_exit)
print("SIGINT handler installed - Ctrl+C will save a checkpoint first")

In [ ]:
# run training with OOM guard + callbacks
trainer = Trainer(
    model=model, args=training_args, train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[MetaCallback(), mid_eval_cb, early_stop_cb, loss_cb],
)
result = train_with_oom_guard(trainer)
print("training done:", result)
trainer.save_model(os.path.join(CKPT_DIR, "final"))
print("final adapter saved to", os.path.join(CKPT_DIR, "final"))